# Remove the chunks that contain no speech
Works on `dataset_clean/` from `03_denoise`, in the run folder that `RUN` in `pipeline_paths.py` sets (or any dataset folder with `metadata.json` + `chunks/`, see `DATASET`).

**Kernel**
- **On the Mac:** the project's `.venv` (`tts (3.12.x)`), which already has `silero-vad`.
- **On the DGX Spark:** `.venv-denoise`, plus `uv pip install --python .venv-denoise/bin/python silero-vad`.

**1. Score** (`outputs/speech_check/<DATASET>_scores.json`)
- Silero VAD gives every 32 ms of every chunk a speech probability.
- A frame counts as speech when that probability is ≥ `THRESHOLD`; the chunk's **speech** is the total time of those frames.
- Takes about 2–5 min for the whole dataset (`WORKERS` threads, each with its own model). It can be resumed.

**2. Review**: each chunk gets one verdict.
- **no speech**: less than `MIN_SPEECH_SEC` of speech.
- **mostly not speech**: speech on less than `MIN_SPEECH_RATIO` of the chunk, e.g. music, noise or silence with a word or two.
- **text longer than speech**: the transcript has more than `MAX_CHARS_PER_SEC` characters per second of speech heard, so the text is probably not all in the audio (Whisper made it up). These are only reported, unless `REMOVE_TEXT_MISMATCH = True`.
- **ok**: everything else.

The cell prints the counts and the worst chunks, and the next cell plays a few of them.

**3. Remove**
- The wavs of the removed chunks move to `<DATASET>/no_speech/`, and `metadata.json` is written without them. Their scores are in `<DATASET>/no_speech.json`.
- Nothing is deleted. The full list before any removal is kept as `<DATASET>/metadata.all.json`.
- You can change the thresholds and run steps 2–3 again: it starts from the full list and moves wavs back when they are kept now.
- To undo everything: set `MIN_SPEECH_SEC = MIN_SPEECH_RATIO = 0` and run steps 2–3 again.

In [ ]:
import json
import os
import re
import shutil
import threading
import time
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import soundfile as sf
import torch
import torchaudio.functional as AF
from silero_vad import load_silero_vad
from tqdm.auto import tqdm

import importlib
import pipeline_paths as P
importlib.reload(P)   # a change of RUN in pipeline_paths.py counts without restarting the kernel

root = P.ROOT
print(P.describe())

In [ ]:
DATASET = P.DATASET_CLEAN   # folder with metadata.json + chunks/ (a path, or a folder name in the project root)

THRESHOLD = 0.5             # Silero speech probability for a 32 ms frame to count as speech (changing it = new scan)
MIN_SPEECH_SEC = 0.3        # less speech than this in a chunk = "no speech"
MIN_SPEECH_RATIO = 0.2      # speech on less than this share of the chunk = "mostly not speech"
MAX_CHARS_PER_SEC = 25.0    # more transcript characters per second of speech = "text longer than speech"
REMOVE_TEXT_MISMATCH = False  # True: remove those too (by default they are only listed)

WORKERS = 4
SAVE_EVERY = 500
N_LISTEN = 8                # chunks to play in the listening cell

In [ ]:
ds = root / DATASET
meta_path, all_path, removed_dir = ds / "metadata.json", ds / "metadata.all.json", ds / "no_speech"
meta = json.load(open(meta_path, encoding="utf-8"))
if all_path.exists():   # an earlier run removed chunks: start from the full list
    full = json.load(open(all_path, encoding="utf-8"))
    extra = {c["file"] for c in meta["chunks"]} - {c["file"] for c in full["chunks"]}
    assert not extra, (f"metadata.json has {len(extra)} chunks that metadata.all.json does not, so the dataset was "
                       f"rebuilt: delete {all_path.name} and {removed_dir.name}/ and run again")
    meta = full
chunks = meta["chunks"]
by_file = {c["file"]: c for c in chunks}


def wav_path(c):
    p = ds / c["file"]
    return p if p.exists() else removed_dir / Path(c["file"]).name


def save_json(path, obj, indent=1):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=indent), encoding="utf-8")
    os.replace(tmp, path)


print(f"{DATASET}: {len(chunks)} chunks, {sum(c['duration'] for c in chunks) / 3600:.2f} h, "
      f"{meta['sample_rate']} Hz; currently in metadata.json: {len(json.load(open(meta_path))['chunks'])}")

## 1. Score every chunk with Silero VAD

In [ ]:
FRAME_SEC = 512 / 16000   # Silero looks at 32 ms frames of 16 kHz audio
scores_path = P.OUTPUTS / "speech_check" / f"{ds.name}_scores.json"
scores = json.load(open(scores_path, encoding="utf-8")) if scores_path.exists() else {}
todo = [c for c in chunks if scores.get(c["file"], {}).get("duration") != c["duration"]
        or scores[c["file"]].get("threshold") != THRESHOLD]
print(f"{len(chunks) - len(todo)} of {len(chunks)} chunks already scored; {len(todo)} to go "
      f"({sum(c['duration'] for c in todo) / 3600:.1f} h of audio)")

local = threading.local()   # one Silero model per thread: the model keeps state between frames


def score(c):
    try:
        if not hasattr(local, "model"):
            local.model = load_silero_vad()
        x, sr = sf.read(wav_path(c), dtype="float32")
        x = AF.resample(torch.from_numpy(x), sr, 16000)
        with torch.no_grad():
            p = local.model.audio_forward(x[None], 16000)[0].numpy()
        return c["file"], dict(duration=c["duration"], threshold=THRESHOLD,
                               speech=round(float((p >= THRESHOLD).sum() * FRAME_SEC), 2),
                               max_prob=round(float(p.max()), 3)), None
    except Exception as e:
        return c["file"], None, repr(e)


torch.set_num_threads(1)
failed = {}
t0 = time.time()
pool = ThreadPoolExecutor(WORKERS)
try:
    for k, (f, s, err) in enumerate(tqdm(pool.map(score, todo), total=len(todo), desc="score"), 1):
        if s is None:
            failed[f] = err
        else:
            scores[f] = s
        if k % SAVE_EVERY == 0:
            save_json(scores_path, scores)
finally:
    pool.shutdown(wait=False, cancel_futures=True)
    save_json(scores_path, scores)
print(f"done in {(time.time() - t0) / 60:.1f} min; {len(failed)} failed")
for f, e in list(failed.items())[:10]:
    print("  ", f, e)

## 2. Review

In [ ]:
def n_chars(text):
    return len(re.sub(r"\W", "", text))


verdict = {}
for c in chunks:
    s = scores.get(c["file"])
    if s is None or s.get("threshold") != THRESHOLD:
        continue
    if s["speech"] < MIN_SPEECH_SEC:
        verdict[c["file"]] = "no speech"
    elif s["speech"] / c["duration"] < MIN_SPEECH_RATIO:
        verdict[c["file"]] = "mostly not speech"
    elif n_chars(c["text"]) / s["speech"] > MAX_CHARS_PER_SEC:
        verdict[c["file"]] = "text longer than speech"
    else:
        verdict[c["file"]] = "ok"
REMOVE = {"no speech", "mostly not speech"} | ({"text longer than speech"} if REMOVE_TEXT_MISMATCH else set())
remove = {f for f, v in verdict.items() if v in REMOVE}

if len(verdict) < len(chunks):
    print(f"{len(chunks) - len(verdict)} chunks are not scored yet: run step 1 again.\n")
print(f"{'':26} {'chunks':>7} {'hours':>7}")
for v in ["no speech", "mostly not speech", "text longer than speech", "ok"]:
    fs = [f for f in verdict if verdict[f] == v]
    mark = "  <- removed" if v in REMOVE else ""
    print(f"{v:26} {len(fs):7d} {sum(by_file[f]['duration'] for f in fs) / 3600:7.2f}{mark}")

if verdict:
    ratio = np.array([scores[f]["speech"] / by_file[f]["duration"] for f in verdict])
    cps = np.array([n_chars(by_file[f]["text"]) / max(scores[f]["speech"], FRAME_SEC) for f in verdict])
    print("\nspeech share of a chunk, percentiles 1/5/25/50:", np.round(np.percentile(ratio, [1, 5, 25, 50]), 2))
    print("characters per second of speech, percentiles 50/95/99:", np.round(np.percentile(cps, [50, 95, 99]), 1))

pod = defaultdict(Counter)
for f in remove:
    pod[by_file[f]["speaker"][:3]][verdict[f]] += 1
if pod:
    print("\nremoved per podcast:")
    for num, cnt in sorted(pod.items(), key=lambda kv: -sum(kv[1].values())):
        print(f"  {num} {meta['podcasts'][num][:50]:50} " + ", ".join(f"{v}: {n}" for v, n in cnt.items()))

for v in ["no speech", "mostly not speech", "text longer than speech"]:
    fs = sorted((f for f in verdict if verdict[f] == v), key=lambda f: scores[f]["speech"] / by_file[f]["duration"])
    if fs:
        print(f"\n{v} ({len(fs)}), the first 12:")
    for f in fs[:12]:
        c, s = by_file[f], scores[f]
        print(f"  {f:32} {c['duration']:5.1f} s, speech {s['speech']:5.1f} s  | {c['text'][:70]}")

### Listen to a few
The chunks that would be removed, least speech first. Each player adds about 1 MB to the notebook.

In [ ]:
from IPython.display import Audio, display

listen = sorted(remove, key=lambda f: scores[f]["speech"] / by_file[f]["duration"])[:N_LISTEN]
for f in listen:
    c = by_file[f]
    print(f"{verdict[f]}: {f}  {c['duration']:.1f} s, speech {scores[f]['speech']:.1f} s  | {c['text']}")
    display(Audio(str(wav_path(c))))

## 3. Remove them from the dataset

In [ ]:
missing = [c["file"] for c in chunks if c["file"] not in verdict]
assert not missing, f"{len(missing)} chunks are not scored: run step 1 again first"
if not all_path.exists():
    shutil.copy2(meta_path, all_path)     # the full list: later runs start from it
removed_dir.mkdir(exist_ok=True)
moved_out = moved_back = 0
for c in chunks:
    here, there = ds / c["file"], removed_dir / Path(c["file"]).name
    if c["file"] in remove and here.exists():
        os.replace(here, there)
        moved_out += 1
    elif c["file"] not in remove and there.exists():
        os.replace(there, here)           # kept now (other thresholds than last time): move it back
        moved_back += 1
kept = [c for c in chunks if c["file"] not in remove]
lost = [c["file"] for c in kept if not (ds / c["file"]).exists()]
assert not lost, f"{len(lost)} kept chunks have no wav: {lost[:5]}"
save_json(ds / "no_speech.json", [dict(c, **scores[c["file"]], verdict=verdict[c["file"]])
                                  for c in chunks if c["file"] in remove])
save_json(meta_path, dict(meta, chunks=kept, speech_check=dict(
    THRESHOLD=THRESHOLD, MIN_SPEECH_SEC=MIN_SPEECH_SEC, MIN_SPEECH_RATIO=MIN_SPEECH_RATIO,
    MAX_CHARS_PER_SEC=MAX_CHARS_PER_SEC, REMOVE_TEXT_MISMATCH=REMOVE_TEXT_MISMATCH, removed=len(remove))), indent=2)
print(f"moved {moved_out} wavs to {removed_dir.name}/, {moved_back} back")
print(f"metadata.json: {len(kept)} chunks, {sum(c['duration'] for c in kept) / 3600:.2f} h "
      f"(removed {len(remove)}, {sum(by_file[f]['duration'] for f in remove) / 60:.1f} min)")
print(f"wavs in chunks/: {len(list((ds / 'chunks').glob('*.wav')))}, in {removed_dir.name}/: {len(list(removed_dir.glob('*.wav')))}")